#### This notebook is used to experiment with DTGraph rules and transformations.

In [4]:
from dtgraph import Neo4jGraph, Rule, Transformation

In [5]:
hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [6]:
from dtgraph.scenarios.movies import Movies

Movies.load(graph)

Flushed database: Deleted 171 nodes, deleted 253 relationships, completed after 16 ms.
Load scenario: Added 171 labels, created 171 nodes, set 564 properties, created 253 relationships, completed after 4 ms.


In [7]:
def reload_dtgraph():
    import importlib
    import dtgraph.parser
    import dtgraph.rule

    importlib.reload(dtgraph.parser)
    importlib.reload(dtgraph.rule)

### Node Rules 

In [1]:
# rule1 = Rule('''
# MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
# GENERATE
# (x = (m):Film {
#     title = m.title
# }),
# (y = (p):Human {
#     name = p.name
# }),
# (x)-[():HAS_ACTOR]->(y),
# (y)-[():ACTED_IN_FILM]->(x)
# ''')


# invalid_chain = Rule('''
# MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
# GENERATE
# (x = (p):Human)-[():REL1]->(y = (m):Film)-[():REL2]->(z = (m):Category)
# ''')


# name = "Human: " + p.name


# generate_humans = Rule('''
# MATCH (p:Person)
# GENERATE
# (x = (p):Human {
#     name = 10 + 10                                                             
# })
# ''')

complex_pattern = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(b:Person)
GENERATE
(x = (a):)-[():WORKED_WITH]->(y = (m):)<-[():WORKED_WITH]-(z = (b):)
''')

complex_pattern1 = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(b:Person)
GENERATE
(x = (a):Actor {
    name = a.name,
    experienced = a.born < 1970
})-[():WORKED_ON {
    movie = m.title,
    since = m.released,
    sameGeneration = a.born < 1970 AND b.born < 1970
}]->(y = (m):Film {
    title = m.title
})<-[():WORKED_ON {
    movie = m.title
}]-(z = (b):Actor {
    name = b.name,
    experienced = b.born < 1970
}),
''')

# Actor and director collaborated on the same movie
rule2 = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Person)-[:ACTED_IN]->(m)
GENERATE
(x = (a):Actor {
    name = a.name
})-[():ACTED_IN {
    movie = m.title,
    year = m.released
}]->(y = (m):Film {
    title = m.title,
    year = m.released
})<-[():DIRECTED {
    movie = m.title
}]-(w = (d):Director {
    name = d.name
})-[():WORKED_WITH {
    movie = m.title
}]->(x)
''')


# A user follows an actor who acted in a movie
rule4 = Rule('''
MATCH (a:Person)-[:FOLLOWS]->(b:Person)-[:ACTED_IN]->(m:Movie)
GENERATE
(x = (a):User {
    name = a.name
})-[():FOLLOWS {
    relation = "social"
}]->(y = (b):Actor {
    name = b.name
})-[():ACTED_IN {
    movie = m.title
}]->(z = (m):Film {
    title = m.title
})
''')

# Reviewer reviewed a movie featuring an actor
rule5 = Rule('''
MATCH (r:Person)-[:REVIEWED]->(m:Movie)<-[:ACTED_IN]-(a:Person)
GENERATE
(x = (r):Reviewer {
    name = r.name
})-[():REVIEWED {
    movie = m.title
}]->(y = (m):Film {
    title = m.title
})<-[():ACTED_IN {
    role = "actor"
}]-(z = (a):Actor {
    name = a.name
})
''')

# Actor worked with a producer on the same movie
rule6 = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:PRODUCED]-(p:Person)
GENERATE
(x = (a):Actor {
    name = a.name
})-[():ACTED_IN {
    movie = m.title
}]->(y = (m):Film {
    title = m.title
})<-[():PRODUCED {
    role = "producer"
}]-(z = (p):Producer {
    name = p.name
})-[():WORKED_WITH {
    movie = m.title,
    relation = "actor-producer"
}]->(x)
''')


# Two actors worked together with age-based comparison
rule7 = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(b:Person)
WHERE id(a) < id(b)
GENERATE
(x = (a):Actor {
    name = a.name,
    born = a.born
})-[():WORKED_WITH {
    movie = m.title,
    olderActor = a.born < b.born,
    ageDifference = b.born - a.born
}]->(z = (b):Actor {
    name = b.name,
    born = b.born
})
''')

# Actor and director collaborated on a movie with enriched context - updated during the meeting (skolems of skolwm nested) - type checking

# Work on skolem of skolem
# Type decleration (inlude bags, sets) and type checking
# axilery functions

rule8 = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Person)
GENERATE
(x = (a):Actor {
    name = a.name,
    experienced = a.born < 1970
})-[():ACTED_IN {
    movie = m.title,
    year = m.released
}]->(y = (m):Film {
    title = m.title,
    year = m.released
})<-[():DIRECTED {
    movie = m.title
}]-(w = (d):Director {
    name = d.name,
    experienced = d.born < 1970
})-[():WORKED_WITH {
    movie = m.title,
    collaboration = "actor-director",
    sameGeneration = a.born < 1970 AND d.born < 1970
}]->(x)
''')


# Actor is connected back to themselves through the movie
rule9 = Rule('''
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
GENERATE
(x = (p):Actor)-[():ACTED_IN {movie = m.title}]->(y = (m):Film)-[():RELATED_TO {reason = "same actor"}]->(x)
''')

# Actor acted in a movie that was directed by a director
rule10 = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Person)
GENERATE
(x = (a):Actor {
    name = a.name,
    born = a.born,
    experienced = a.born < 1970
})-[():ACTED_IN {
    movie = m.title
}]->(y = (m):Film {
    title = m.title,
    year = m.released
})-[():DIRECTED_BY {
    movie = m.title
}]->(z = (d):Director {
    name = d.name,
    born = d.born
})
''')

NameError: name 'Rule' is not defined

### Execute Rules

In [ ]:
my_transform = Transformation([rule10])
my_transform.apply_on(graph)

### Abort Transformation

In [ ]:
my_transform.abort()